# U-Net Data Tensors from CRIMAC Zarr

Continues from **notebook 06** (zarr parsing + bottom detection) to produce
and persist the actual training tensors consumed by the U-Net model.

**Pipeline summary:**
1. Open a CRIMAC-preprocessor zarr store  
2. Run CRIMAC ensemble bottom detection  
3. Build binary segmentation masks ("below seafloor")  
4. Extract fixed-width (image, label) patches  
5. Save all patches to a compressed NumPy archive (`.npz`)  
6. Wrap the archive in a `ZarrBottomDataset` (PyTorch Dataset)  
7. Create train / validation DataLoaders  
8. Verify tensor shapes and dtype before model ingestion  


In [ ]:
import sys
sys.path.insert(0, '../../..')

from pathlib import Path
import numpy as np

from raw_sensor_integration.zarr_dataset import save_patches_npz, ZarrBottomDataset


## 1. Configuration

Set `ZARR_PATH` to the `.zarr` directory produced by the CRIMAC preprocessor
and `OUTPUT_DIR` to where the `.npz` patch archive should be saved.


In [ ]:
import os

# ── paths ────────────────────────────────────────────────────────────
zarr_env = os.getenv("CRIMAC_ZARR_PATH", "")
if zarr_env:
    ZARR_PATH = Path(zarr_env).expanduser()
else:
    # Set CRIMAC_ZARR_PATH in your environment or replace this placeholder.
    ZARR_PATH = Path("/Users/axelhagen/Downloads/crimac_output/test.zarr")

output_dir_env = os.getenv("UNET_TENSORS_OUTPUT_DIR", "tensors")
OUTPUT_DIR = Path(output_dir_env).expanduser()
output_npz_env = os.getenv("UNET_TENSORS_OUTPUT_NPZ", "")
OUTPUT_NPZ = Path(output_npz_env).expanduser() if output_npz_env else OUTPUT_DIR / "bottom_patches.npz"

if not ZARR_PATH.exists():
    raise FileNotFoundError(
        "ZARR_PATH does not exist. Set CRIMAC_ZARR_PATH or update ZARR_PATH in this cell. "
        f"Current value: {ZARR_PATH}"
    )

# ── patch settings ───────────────────────────────────────────────────
PATCH_WIDTH = 256   # pings per patch  (W)
STRIDE = 128        # overlap = PATCH_WIDTH - STRIDE

# ── Sv normalisation range (dB re 1 m⁻¹) ────────────────────────────
VMIN = -80.0
VMAX = -34.0


## 2. Save Patches to Disk

This cell runs bottom detection, extracts all valid patches, and writes
them to a single compressed `.npz` file.

The archive contains:

| key | shape | dtype | description |
|---|---|---|---|
| `images` | (N, C, H, W) | float32 | normalised Sv, values in [0, 1] |
| `labels` | (N, H, W) | uint8 | binary "below seafloor" mask |
| `frequencies` | (C,) | float64 | Hz |
| `range_m` | (H,) | float64 | range from transducer (metres) |
| `vmin` / `vmax` | scalar | float64 | normalisation bounds |
| `patch_width` / `stride` | scalar | int64 | window settings |


In [ ]:
n_saved = save_patches_npz(
    zarr_path=ZARR_PATH,
    output_npz=OUTPUT_NPZ,
    patch_width=PATCH_WIDTH,
    stride=STRIDE,
    vmin=VMIN,
    vmax=VMAX,
    require_bottom=True,
    verbose=True,
)
print(f'\nArchive size: {OUTPUT_NPZ.stat().st_size / 1e6:.1f} MB')


## 3. Create ZarrBottomDataset

`ZarrBottomDataset` wraps the `.npz` archive and exposes a PyTorch
`Dataset` interface. Each call to `__getitem__` returns an
`(image, label)` pair of `torch.Tensor` objects.


In [ ]:
dataset = ZarrBottomDataset(OUTPUT_NPZ)
print(dataset)
print(f'  n_samples  : {len(dataset)}')

img, lbl = dataset[0]
print(f'  image shape: {tuple(img.shape)}  dtype={img.dtype}')
print(f'  label shape: {tuple(lbl.shape)}  dtype={lbl.dtype}')
print(f'  image range: [{float(img.min()):.3f}, {float(img.max()):.3f}]')
print(f'  label unique values: {lbl.unique().tolist()}')


## 4. Train / Validation Split

To avoid leakage when `STRIDE < PATCH_WIDTH`, split by deterministic,
non-overlapping survey regions first, then extract train/val patches only
from their assigned regions.

In [ ]:
import torch
import xarray as xr
from torch.utils.data import DataLoader

BATCH_SIZE = 8
VAL_FRAC = 0.2
SEED = 42

def compute_non_overlapping_region_starts(n_pings, patch_width, stride, region_patches=8):
    """Build disjoint patch-start regions with a guard gap to prevent overlap across regions."""
    if patch_width <= 0 or stride <= 0:
        raise ValueError("patch_width and stride must be positive")
    max_start = n_pings - patch_width
    if max_start < 0:
        return []

    guard = max(0, patch_width - stride)
    region_span = patch_width + (region_patches - 1) * stride

    regions = []
    region_start = 0
    while region_start <= max_start:
        region_end = min(n_pings, region_start + region_span)
        last_start = region_end - patch_width
        starts = list(range(region_start, last_start + 1, stride))
        if starts:
            regions.append(starts)
        region_start = region_end + guard

    return regions

with xr.open_zarr(ZARR_PATH, consolidated=False) as ds_meta:
    n_pings = int(ds_meta.sizes["ping_time"])

regions = compute_non_overlapping_region_starts(n_pings, PATCH_WIDTH, STRIDE)
n_regions = len(regions)
generator = torch.Generator().manual_seed(SEED)

if n_regions >= 2:
    perm = torch.randperm(n_regions, generator=generator).tolist()
    n_val_regions = max(1, int(round(n_regions * VAL_FRAC)))
    n_val_regions = min(n_regions - 1, n_val_regions)
    val_region_ids = set(perm[:n_val_regions])

    train_starts = [
        s
        for rid, starts in enumerate(regions)
        if rid not in val_region_ids
        for s in starts
    ]
    val_starts = [
        s
        for rid, starts in enumerate(regions)
        if rid in val_region_ids
        for s in starts
    ]
    split_note = f"region split: total={n_regions}, train={n_regions - n_val_regions}, val={n_val_regions}"
else:
    all_starts = list(range(0, n_pings - PATCH_WIDTH + 1, STRIDE))
    n_patches = len(all_starts)
    if n_patches < 2:
        raise ValueError(
            "Not enough patches for train/val split. "
            f"Found {n_patches} patch(es) with n_pings={n_pings}, patch_width={PATCH_WIDTH}, stride={STRIDE}. "
            "Try reducing patch_width or stride."
        )

    perm = torch.randperm(n_patches, generator=generator).tolist()
    n_val = max(1, int(round(n_patches * VAL_FRAC)))
    n_val = min(n_patches - 1, n_val)
    val_ids = set(perm[:n_val])
    val_starts = [s for i, s in enumerate(all_starts) if i in val_ids]
    train_starts = [s for i, s in enumerate(all_starts) if i not in val_ids]
    split_note = f"fallback patch split: total={n_patches}, train={len(train_starts)}, val={len(val_starts)}"

train_npz = OUTPUT_DIR / "bottom_patches_train.npz"
val_npz = OUTPUT_DIR / "bottom_patches_val.npz"

save_patches_npz(
    zarr_path=ZARR_PATH,
    output_npz=train_npz,
    patch_width=PATCH_WIDTH,
    stride=STRIDE,
    vmin=VMIN,
    vmax=VMAX,
    allowed_patch_starts=train_starts,
    require_bottom=True,
    verbose=True,
)

save_patches_npz(
    zarr_path=ZARR_PATH,
    output_npz=val_npz,
    patch_width=PATCH_WIDTH,
    stride=STRIDE,
    vmin=VMIN,
    vmax=VMAX,
    allowed_patch_starts=val_starts,
    require_bottom=True,
    verbose=True,
)

train_set = ZarrBottomDataset(train_npz)
val_set = ZarrBottomDataset(val_npz)
n_train = len(train_set)
n_val = len(val_set)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

print(split_note)
print(f"Train batches: {len(train_loader)}  ({n_train} patches)")
print(f"Val   batches: {len(val_loader)}  ({n_val} patches)")


## 5. Verify One Batch

Confirm that the tensors coming out of the DataLoader have the correct
shape and dtype for a U-Net that expects `(B, C, H, W)` float32 inputs
and `(B, H, W)` float32 segmentation targets.


In [ ]:
images_batch, labels_batch = next(iter(train_loader))

print("One batch from train_loader:")
print(f'  images : {tuple(images_batch.shape)}  dtype={images_batch.dtype}')
print(f'  labels : {tuple(labels_batch.shape)}  dtype={labels_batch.dtype}')
print(f'  image  range: [{float(images_batch.min()):.3f}, {float(images_batch.max()):.3f}]')
print(f'  label  unique values: {labels_batch.unique().tolist()}')
assert images_batch.dtype == torch.float32, "images must be float32"
assert labels_batch.dtype == torch.float32, "labels must be float32"
assert images_batch.ndim == 4, "images must be (B, C, H, W)"
assert labels_batch.ndim == 3, "labels must be (B, H, W)"
print("\nAll assertions passed — tensors are ready for U-Net ingestion.")


## 6. Preview Patches

Visual sanity check: show image + mask side-by-side for the first few
patches in the training set.


In [ ]:
import matplotlib.pyplot as plt

N_PREVIEW = min(4, n_train)
fig, axes = plt.subplots(2, N_PREVIEW, figsize=(5 * N_PREVIEW, 8), squeeze=False)

for i in range(N_PREVIEW):
    img_i, lbl_i = train_set[i]
    # Use first channel for display (works for single- and multi-freq)
    axes[0, i].imshow(img_i[0].numpy(), aspect='auto', cmap='viridis',
                      vmin=0, vmax=1, origin='upper')
    axes[0, i].set_title(f'Patch {i} — Sv')
    axes[0, i].set_xlabel('Ping')
    axes[0, i].set_ylabel('Range sample')

    axes[1, i].imshow(lbl_i.numpy(), aspect='auto', cmap='gray_r',
                      vmin=0, vmax=1, origin='upper')
    axes[1, i].set_title(f'Patch {i} — mask')
    axes[1, i].set_xlabel('Ping')
    axes[1, i].set_ylabel('Range sample')

plt.suptitle(f'Training patches  ({PATCH_WIDTH} pings wide, stride {STRIDE})')
plt.tight_layout()
plt.show()


## 7. Multi-Frequency Tensors

If the zarr contains more than one frequency, pass all of them to
`save_patches_npz` via `frequencies_hz`. The resulting image tensors
will have shape `(B, n_freq, H, W)`.


In [ ]:
import xarray as xr

with xr.open_zarr(ZARR_PATH, consolidated=False) as ds_meta:
    all_freqs = list(ds_meta.frequency.values)
print(f"All frequencies in zarr: {[f/1000 for f in all_freqs]} kHz")

if len(all_freqs) > 1:
    mf_npz = OUTPUT_DIR / 'bottom_patches_multifreq.npz'
    n_mf = save_patches_npz(
        zarr_path=ZARR_PATH,
        output_npz=mf_npz,
        frequencies_hz=all_freqs,
        patch_width=PATCH_WIDTH,
        stride=STRIDE,
        vmin=VMIN,
        vmax=VMAX,
        require_bottom=True,
        verbose=True,
    )
    mf_dataset = ZarrBottomDataset(mf_npz)
    print(mf_dataset)
    mf_img, mf_lbl = mf_dataset[0]
    print(f'  image shape: {tuple(mf_img.shape)}  (n_freq, H, W)')
else:
    print("Only one frequency present — multi-frequency example skipped.")


## Summary

| Step | Output |
|------|--------|
| `save_patches_npz()` | `tensors/bottom_patches.npz` — compressed patch archive |
| `ZarrBottomDataset()` | PyTorch Dataset — `(C, H, W)` image + `(H, W)` label |
| `DataLoader` | `(B, C, H, W)` batches ready for U-Net training |

Feed `images_batch` directly into a U-Net encoder and use `labels_batch`
as the segmentation target (binary cross-entropy or dice loss).
